In [2]:
# Import Pandas
import pandas as pd

#Load Data
data = pd.read_csv('/content/tweets.csv')
data

,id,label,tweet
0,1,0,#fingerprint #Pregnancy Test https://goo.gl/h1...
1,2,0,Finally a transparant silicon case ^^ Thanks t...
2,3,0,We love this! Would you go? #talk #makememorie...
3,4,0,I'm wired I know I'm George I was made that wa...
4,5,1,What amazing service! Apple won't even talk to...
...,...,...,...
7915,7916,0,Live out loud #lol #liveoutloud #selfie #smile...
7916,7917,0,We would like to wish you an amazing day! Make...
7917,7918,0,Helping my lovely 90 year old neighbor with he...
7918,7919,0,Finally got my #smart #pocket #wifi stay conne...


In [3]:
data.info() # Data info

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7920 entries, 0 to 7919
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      7920 non-null   int64 
 1   label   7920 non-null   int64 
 2   tweet   7920 non-null   object
dtypes: int64(2), object(1)
memory usage: 185.8+ KB


In [4]:
df=data.drop("id",axis=1) #drop the id column

In [5]:
# Import pandas,train_test_split,CountVectorizer,MultinomialNB,accuracy_score, classification_report, confusion_matrix
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [6]:
import pandas as pd
import re # for replacing
import nltk # natural language tool kit
from nltk.corpus import stopwords # Importing stopwords

nltk.download('stopwords') # downloading stop words
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower() # converting to Lowercase
    text = re.sub(r'http\S+|@\w+|#\w+', '', text) # Removing http,@,# char and replace it with ''
    text = re.sub(r'[^a-z\s]', '', text) # only keep letters and replace all others with ''
    words = text.split() # splitting the words
    words = [w for w in words if w not in stop_words] # will not include stopwords
    return " ".join(words)

df['clean_text'] = df['tweet'].apply(clean_text)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [7]:
#df['label'] = df['label'].replace(1, 'Negative') # Replacing label 1 with Negative
#df['label'] = df['label'].replace(0, 'Positive') # Replacing label  with Negative
df["label"].value_counts()

,count
label,
0,5894
1,2026


In [8]:
df

,label,tweet,clean_text
0,0,#fingerprint #Pregnancy Test https://goo.gl/h1...,test
1,0,Finally a transparant silicon case ^^ Thanks t...,finally transparant silicon case thanks uncle
2,0,We love this! Would you go? #talk #makememorie...,love would go
3,0,I'm wired I know I'm George I was made that wa...,im wired know im george made way
4,1,What amazing service! Apple won't even talk to...,amazing service apple wont even talk question ...
...,...,...,...
7915,0,Live out loud #lol #liveoutloud #selfie #smile...,live loud
7916,0,We would like to wish you an amazing day! Make...,would like wish amazing day make every minute ...
7917,0,Helping my lovely 90 year old neighbor with he...,helping lovely year old neighbor ipad morning ...
7918,0,Finally got my #smart #pocket #wifi stay conne...,finally got stay connected anytimeanywhere


In [9]:
# Import CountVectorizer and  numpy
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(df['clean_text']) # transforming the cleaned data

co_matrix = (X.T @ X) # association btw the words
co_matrix.setdiag(0)

words = vectorizer.get_feature_names_out() # get feature names
co_df = pd.DataFrame(co_matrix.toarray(), index=words, columns=words)

print(co_df.head())

             aaaahhhhhhh  aah  aalborg  aand  aapl  aaron  aarrrggghhhh  aas  \
aaaahhhhhhh            0    0        0     0     0      0             0    0   
aah                    0    0        0     0     0      0             0    0   
aalborg                0    0        0     0     0      0             0    0   
aand                   0    0        0     0     0      0             0    0   
aapl                   0    0        0     0     0      0             0    0   

             aashamsakal  ab  ...  zoo  zoom  zoooom  \
aaaahhhhhhh            0   0  ...    0     0       0   
aah                    0   0  ...    0     0       0   
aalborg                0   0  ...    0     0       0   
aand                   0   0  ...    0     0       0   
aapl                   0   0  ...    0     0       0   

             zpictwittercomgskxbmkrto  zpictwittercomhnzltjvn  \
aaaahhhhhhh                         0                       0   
aah                                 0               

In [10]:
print(co_df['angry'].sort_values(ascending=False).head(10)) # finding similar words to angry

work       2
apple      2
ok         2
phone      2
iphone     2
birds      2
got        2
another    1
playing    1
take       1
Name: angry, dtype: int64


In [11]:
print(co_df['happy'].sort_values(ascending=False).head(10)) # finding similar words to happy

birthday    39
new         39
day         34
im          25
got         16
year        13
iphone      13
phone       11
apple        8
game         8
Name: happy, dtype: int64


In [12]:
# importing TfidfVectorizer,train_test_split,LogisticRegression,SMOTE
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE


tfidf = TfidfVectorizer(max_features=5000) # vectorizer is used to convert words to numbers

X = tfidf.fit_transform(df['clean_text'])
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2) # Splitting the data to train and test
smote = SMOTE(sampling_strategy='minority', random_state=42)# Appling smote
X_train, y_train= smote.fit_resample(X_train, y_train)
y_train.value_counts()

model = LogisticRegression(max_iter=200) # Logistic Regression model
model.fit(X_train, y_train ) # Training the model

print(model.score(X_test, y_test))

0.8491161616161617


In [13]:
import nltk #library # natural language tool kit
import numpy as np
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet #stopwords
import nltk

nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt_tab')
nltk.download('stopwords')
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))
mapping = {0: "positive", 1: "negative"}


new_text = "I am so happy today"


text = new_text.lower()  # 1. Lowercase
text = re.sub(r'[^a-z\s]', '', text)  # 2. Remove punctuation/numbers
words = text.split()     # 3. Tokenize
print(words)
words = [w for w in words if w not in stop_words]  # 4. Remove stopwords
words = [lemmatizer.lemmatize(w,pos="v") for w in words]   # 5. Lemmatization

# transform using SAME tfidf
new_vec = tfidf.transform(words)

# predict
prediction = model.predict(new_vec)
import numpy as np
pred_class = np.argmax(prediction)

print("Predicted Class:", pred_class)
#label = le.inverse_transform([pred_class])
label = ["positive", "negative"]
print(label[pred_class])



[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


['i', 'am', 'so', 'happy', 'today']
Predicted Class: 0
positive


In [14]:
# BART
from transformers import pipeline # Imported pipeline

classifier = pipeline(
    "text-classification", # To Text Classification
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

print(classifier("I don't Like this Laptop!"))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'NEGATIVE', 'score': 0.9927581548690796}]


In [15]:
# Tokenization + Padding
from tensorflow.keras.preprocessing.text import Tokenizer #imported Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences #imported pad_Sequence

tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(df['clean_text'])

seqs = tokenizer.texts_to_sequences(df['clean_text']) # Text to seq
X_pad = pad_sequences(seqs, maxlen=100) # maxlength for each tweet

In [16]:
# 1.RNN
from tensorflow.keras.models import Sequential # imported Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense # imported Embedding, SimpleRNN, Dense

model = Sequential([
    Embedding(5000, 128),
    SimpleRNN(64),
    Dense(1, activation='sigmoid')
])

In [17]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(df['clean_text'])

X = tokenizer.texts_to_sequences(df['clean_text'])
X = pad_sequences(X, maxlen=100)

In [18]:
#from sklearn.preprocessing import LabelEncoder # imported  LabelEncoder

#le = LabelEncoder()
y = df['label'] # Encoded the Y

print(y[:5])

0    0
1    0
2    0
3    0
4    1
Name: label, dtype: int64


In [19]:
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(X, y, epochs=5)

Epoch 1/5
248/248 ━━━━━━━━━━━━━━━━━━━━ 10s 30ms/step - accuracy: 0.8365 - loss: 0.3602
Epoch 2/5
248/248 ━━━━━━━━━━━━━━━━━━━━ 10s 29ms/step - accuracy: 0.9384 - loss: 0.1752
Epoch 3/5
248/248 ━━━━━━━━━━━━━━━━━━━━ 9s 35ms/step - accuracy: 0.9744 - loss: 0.0782
Epoch 4/5
248/248 ━━━━━━━━━━━━━━━━━━━━ 9s 35ms/step - accuracy: 0.9889 - loss: 0.0407
Epoch 5/5
248/248 ━━━━━━━━━━━━━━━━━━━━ 13s 54ms/step - accuracy: 0.9937 - loss: 0.0241


In [20]:
new_text = ["I am happy today."]


# step 1: tokenize
seq = tokenizer.texts_to_sequences(new_text)


# step 2: pad
padded = pad_sequences(seq, maxlen=100)

# step 3: predict
pred = model.predict(padded)

print("Predicted Class :",pred)
#label = le.inverse_transform([pred_class])
print(label[pred_class])


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 306ms/step
Predicted Class : [[0.00015774]]
positive


In [21]:
print(np.unique(y_train, return_counts=True))

(array([0, 1]), array([4704, 4704]))


In [22]:
import numpy as np

pred_class = np.argmax(pred)
print(pred_class)

0


In [24]:
label = label[pred_class]
print("Predicted Class :",label)

Predicted Class : positive


In [61]:
#2.lstm

from tensorflow.keras.layers import LSTM

model = Sequential([
    Embedding(5000, 128),
    LSTM(64),
    Dense(1, activation='sigmoid')
])

In [62]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 20% test data
    random_state=42     # reproducibility
)
smote = SMOTE(sampling_strategy='minority', random_state=42)# Appling smote
X_train, y_train= smote.fit_resample(X_train, y_train)


In [63]:
model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.fit(X_train, y_train, epochs=5, validation_data=(X_test, y_test))

Epoch 1/5
297/297 ━━━━━━━━━━━━━━━━━━━━ 26s 80ms/step - accuracy: 0.7957 - loss: 0.4588 - val_accuracy: 0.8157 - val_loss: 0.4106
Epoch 2/5
297/297 ━━━━━━━━━━━━━━━━━━━━ 38s 70ms/step - accuracy: 0.8605 - loss: 0.3444 - val_accuracy: 0.8491 - val_loss: 0.3530
Epoch 3/5
297/297 ━━━━━━━━━━━━━━━━━━━━ 41s 71ms/step - accuracy: 0.8975 - loss: 0.2694 - val_accuracy: 0.8277 - val_loss: 0.3819
Epoch 4/5
297/297 ━━━━━━━━━━━━━━━━━━━━ 40s 66ms/step - accuracy: 0.9271 - loss: 0.2024 - val_accuracy: 0.8074 - val_loss: 0.4945
Epoch 5/5
297/297 ━━━━━━━━━━━━━━━━━━━━ 21s 72ms/step - accuracy: 0.9487 - loss: 0.1492 - val_accuracy: 0.7721 - val_loss: 0.6354


In [66]:

new_text = ["I feel so happy and excited"]

seq = tokenizer.texts_to_sequences(new_text)
padded = pad_sequences(seq, maxlen=100)

pred = model.predict(padded)


pred_class = int(pred[0][0] > 0.5)

print("Predicted Class:", pred_class)

predicted_label = label[pred_class]
print("Predicted Label:", predicted_label)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
Predicted Class: 0
Predicted Label: p


In [56]:
#  GRU

from tensorflow.keras.layers import GRU

model = Sequential([
    Embedding(5000, 128),
    GRU(64),
    Dense(1, activation='sigmoid')
])

In [57]:
model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.fit(X_train, y_train, epochs=5, validation_data=(X_test, y_test))

Epoch 1/5
297/297 ━━━━━━━━━━━━━━━━━━━━ 35s 89ms/step - accuracy: 0.8002 - loss: 0.4553 - val_accuracy: 0.8359 - val_loss: 0.3673
Epoch 2/5
297/297 ━━━━━━━━━━━━━━━━━━━━ 22s 75ms/step - accuracy: 0.8667 - loss: 0.3338 - val_accuracy: 0.8371 - val_loss: 0.3812
Epoch 3/5
297/297 ━━━━━━━━━━━━━━━━━━━━ 23s 79ms/step - accuracy: 0.9045 - loss: 0.2483 - val_accuracy: 0.8112 - val_loss: 0.4519
Epoch 4/5
297/297 ━━━━━━━━━━━━━━━━━━━━ 27s 90ms/step - accuracy: 0.9349 - loss: 0.1816 - val_accuracy: 0.7917 - val_loss: 0.5275
Epoch 5/5
297/297 ━━━━━━━━━━━━━━━━━━━━ 27s 89ms/step - accuracy: 0.9535 - loss: 0.1345 - val_accuracy: 0.7860 - val_loss: 0.6681


In [59]:
new_text = ["i am so happy today"]

seq = tokenizer.texts_to_sequences(new_text)
padded = pad_sequences(seq, maxlen=100)

pred = model.predict(padded)


pred_class = int(pred[0][0] > 0.5)

print("Predicted Class:", pred_class)

predicted_label = labels[pred_class]
print("Predicted Label:", predicted_label)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 144ms/step
Predicted Class: 0
Predicted Label: p
